# 03 · Execute the Cortex-M4 firmware in Renode

This notebook measures corrections inside compiled bare-metal ARM firmware. The benchmark injects recorded samples with controlled delays, verifies retained scores against the dense oracle, records instructions between two CPU hooks, and measures repair-stack use with a canary.

Set `TINYML_NOTEBOOK_REBUILD=1` and make the Arm GCC and Renode executables available to repeat every emulated case. The included weights and short attributed clips run offline.

In [1]:
import json
import os
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "study.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from tinyml.config import DATASETS, RESULTS, SEEDS, run_dir

REBUILD = os.environ.get("TINYML_NOTEBOOK_REBUILD") == "1"
DATA = Path(os.environ.get("TINYML_DATA_DIR", ROOT / "data")).resolve()
WORKERS = os.environ.get("TINYML_NOTEBOOK_WORKERS", "2")
SCRATCH = ROOT / "build" / "notebooks"

def run(*args):
    command = [sys.executable, *map(str, args)]
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
    if result.returncode:
        print(result.stdout + result.stderr)
        result.check_returncode()
    return result.stdout + result.stderr

print("Execution mode:", "full reproduction" if REBUILD else "inspect archived outputs")
print("Models:", len(DATASETS) * len(SEEDS), "| seeds:", ", ".join(map(str, SEEDS)))

Execution mode: full reproduction
Models: 6 | seeds: 13, 37, 73


In [2]:
import shutil
from tinyml.analysis import paired, principal
from tinyml.counter_check import EXPECTED

counter = json.loads((RESULTS / "counter_check.json").read_text())
assert counter["expected"] == counter["observed"] == EXPECTED
if REBUILD:
    print(run("-m", "tinyml.counter_check"), end="")
    for dataset in DATASETS:
        for seed in SEEDS:
            source = run_dir(RESULTS, dataset, seed)
            target = run_dir(SCRATCH, dataset, seed)
            target.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source / "int8.npz", target / "int8.npz")
    print(run("-m", "tinyml.evaluate", "--backend", "renode", "--results", SCRATCH,
              "--build", SCRATCH / "arm_build", "--workers", WORKERS), end="")
    for dataset in DATASETS:
        for seed in SEEDS:
            filename = "renode_events.csv.gz"
            actual = pd.read_csv(run_dir(SCRATCH, dataset, seed) / filename)
            expected = pd.read_csv(run_dir(RESULTS, dataset, seed) / filename)
            pd.testing.assert_frame_equal(actual, expected)
    print("All six regenerated Renode event tables match the archived rows.")
else:
    display(pd.Series(counter, name="Recorded counter validation"))

{
  "expected": 11,
  "observed": 11,
  "compiler": "arm-none-eabi-gcc (15:14.2.rel1-1) 14.2.1 20241119",
  "simulator": "Renode v1.17.0"
}


renode wisdm/37 horizon=48: 306 cases × 8 methods
renode wisdm/13 horizon=48: 306 cases × 8 methods
renode wisdm/37 horizon=8: 20 cases × 8 methods
renode wisdm/13 horizon=8: 20 cases × 8 methods
renode wisdm/37 horizon=24: 20 cases × 8 methods
renode wisdm/13 horizon=24: 20 cases × 8 methods
renode wisdm/73 horizon=48: 306 cases × 8 methods
renode mhealth/13 horizon=48: 204 cases × 8 methods
renode wisdm/73 horizon=8: 20 cases × 8 methods
renode mhealth/13 horizon=8: 20 cases × 8 methods
renode wisdm/73 horizon=24: 20 cases × 8 methods
renode mhealth/13 horizon=24: 20 cases × 8 methods
renode mhealth/37 horizon=48: 204 cases × 8 methods
renode mhealth/73 horizon=48: 204 cases × 8 methods
renode mhealth/37 horizon=8: 20 cases × 8 methods
renode mhealth/37 horizon=24: 20 cases × 8 methods
renode mhealth/73 horizon=8: 20 cases × 8 methods
renode mhealth/73 horizon=24: 20 cases × 8 methods


All six regenerated Renode event tables match the archived rows.


In [3]:
records = []
for dataset in DATASETS:
    for seed in SEEDS:
        records.append(pd.read_csv(run_dir(RESULTS, dataset, seed) / "renode_events.csv.gz"))
arm = pd.concat(records, ignore_index=True)
assert arm["mismatches"].eq(0).all()
display(arm.groupby(["dataset", "horizon"]).agg(
    trials=("instructions", "size"), score_pairs=("score_pairs_checked", "sum"),
    mismatches=("mismatches", "sum"), largest_measured_sram=("application_sram_bytes", "max"),
))

trials  score_pairs  mismatches  largest_measured_sram
dataset horizon                                                        
mhealth 8           480       103680           0                   1212
        24          480       288000           0                   2100
        48         4896      5757696           0                   3420
wisdm   8           480       155520           0                   1266
        24          480       432000           0                   2250
        48         7344     12954816           0                   3714

In [4]:
best = paired(principal(arm), "instructions", "adaptive")
savings = best.groupby(["dataset", "seed"])["reduction_pct"].median()
display(savings.unstack("seed").round(2))
display(savings.groupby("dataset").agg(["mean", "std"]).round(2))

seed,13,37,73
dataset,,,
mhealth,36.17,34.76,35.96
wisdm,29.22,27.74,26.84


,mean,std
dataset,,
mhealth,35.63,0.76
wisdm,27.93,1.20


Renode reports executed instructions. Device latency and energy remain measurements for physical hardware. The SRAM number is the live C state plus observed correction stack watermark.